# Stage 2 — Subtask 4: Preprocessing Pipeline

This notebook:
1. Documents and justifies the preprocessing pipeline
2. Computes dataset-specific normalisation statistics from the training split
3. Exports `normalisation_stats.json` — **import this in all training scripts**


In [14]:
import json, sys
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt


from config_stage2 import *

PREPROC_OUT.mkdir(parents=True, exist_ok=True)

## 1. Pipeline Specification

### Why 256 → 224?
- Standard ViT-B/16 and ViT-H/14 preprocessing (Dosovitskiy et al. 2020; Assran et al. 2023)
- At patch size 16, 224×224 produces exactly **196 patches (14×14)** — matches positional embedding size

### Why NOT ImageNet normalisation?
PlantVillage images are dominated by chlorophyll green. ImageNet (R:0.485, G:0.456, B:0.406) assumes balanced RGB from natural scene photography. The green channel (the primary disease deviation signal) would be systematically mis-normalised.

### Critical Rule
Statistics are computed **from the training split only**. Applying training stats to val/test is correct (they normalise by the training distribution). Computing separate stats per split would be data leakage.


In [15]:
pipeline_spec = {
    "target_resolution": {"resize": IMAGE_RESIZE, "crop": IMAGE_CROP},
    "normalisation_method": "Dataset-specific: computed from PlantVillage training split ONLY",
    "split_protocol": {
        "method": "Stratified random split",
        "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
        "seed": RANDOM_SEED,
        "why_stratified": (
            "Preserves class distribution across all splits. Non-stratified splitting "
            "risks zero test samples for rare classes, making those classes unmeasurable."
        ),
    },
    "why_not_imagenet_stats": (
        "PlantVillage mean G >> ImageNet mean G (dominated by chlorophyll). "
        "Using ImageNet stats over-normalises the green channel — "
        "exactly the channel most diagnostic for disease."
    ),
}

pipeline_spec_path = PREPROC_OUT / "pipeline_specification.json"
with open(pipeline_spec_path, "w") as f:
    json.dump(pipeline_spec, f, indent=2)
print("Pipeline specification saved.")


Pipeline specification saved.


## 2. Compute Normalisation Statistics

Run this with the PlantVillage dataset present. Uses Welford online algorithm for numerically stable computation over large datasets without loading everything into RAM.


In [16]:
def compute_normalisation_stats(paths: list, target: tuple = (IMAGE_CROP, IMAGE_CROP),
                                 sample_limit: int = 10000) -> dict:
    np.random.seed(RANDOM_SEED)
    if len(paths) > sample_limit:
        paths = list(np.random.choice(paths, sample_limit, replace=False))
    n_pixels = 0
    ch_sum = np.zeros(3, dtype=np.float64)
    ch_sq  = np.zeros(3, dtype=np.float64)
    for path in tqdm(paths, desc="Computing normalisation stats"):
        try:
            arr = np.array(Image.open(path).convert("RGB").resize(target),
                           dtype=np.float32) / 255.0
            h, w, c = arr.shape; n = h * w; n_pixels += n
            flat = arr.reshape(-1, c)
            ch_sum += flat.sum(axis=0)
            ch_sq  += (flat ** 2).sum(axis=0)
        except Exception:
            continue
    if n_pixels == 0:
        return {"mean": [0.485,0.456,0.406], "std": [0.229,0.224,0.225],
                "warning": "PLACEHOLDER — dataset not found. Computed 0 images."}
    mean = ch_sum / n_pixels
    std  = np.sqrt(np.clip(ch_sq / n_pixels - mean**2, 1e-8, None))
    return {
        "mean": [round(float(m),6) for m in mean],
        "std":  [round(float(s),6) for s in std],
        "n_images": len(paths), "n_pixels": int(n_pixels),
        "comparison": {
            "imagenet_mean": [0.485,0.456,0.406],
            "plantvillage_mean": [round(float(m),6) for m in mean],
            "delta": [round(float(m-im),4) for m,im in zip(mean,[0.485,0.456,0.406])],
            "note": "Positive G-channel delta confirms PlantVillage is greener than ImageNet.",
        },
    }

# Check for pre-saved training paths from ST6
split_path = OUTPUTS_DIR / "splits" / "train_split.json"

split_data = json.loads(split_path.read_text())
train_paths = [Path(p) for p in split_data["paths"]]
print(f"Using {len(train_paths):,} training paths from SubT4 label splits")

stats = compute_normalisation_stats(train_paths)
print(f"\nMean (R,G,B): {stats['mean']}")
print(f"Std  (R,G,B): {stats['std']}")
if "warning" in stats:
    print(f"\n⚠️  {stats['warning']}")
else:
    d = stats["comparison"]["delta"]
    print(f"Delta vs ImageNet (R,G,B): {d}")
    if d[1] > 0:
        print("✅ G channel is higher than ImageNet — confirms dataset-specific stats are needed")

stats_path = PREPROC_OUT / "normalisation_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"\n✅ Normalisation stats → {stats_path}")
print("ACTION: Update config.py NORM_MEAN and NORM_STD with the values above.")


Using 38,013 training paths from ST6 splits


Computing normalisation stats: 100%|██████████| 10000/10000 [01:50<00:00, 90.35it/s]


Mean (R,G,B): [0.466726, 0.488969, 0.41028]
Std  (R,G,B): [0.195034, 0.170282, 0.213409]
Delta vs ImageNet (R,G,B): [-0.0183, 0.033, 0.0043]
✅ G channel is higher than ImageNet — confirms dataset-specific stats are needed

✅ Normalisation stats → D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\preprocessing\normalisation_stats.json
ACTION: Update config.py NORM_MEAN and NORM_STD with the values above.
